# MACHINE LEARNING PROJECT

### Track T4 – Semi-Supervised Learning (SSL)
The objective of the project is to study and apply semi-supervised learning techniques to tabular data, analyzing how the limited availability of labels affects the performance of predictive models.

I'm interested in cybersecurity, so I chose the CSE-CIC-IDS2018 Intrusion CSVs (IDS 2018) dataset for the classification problem.

The dataset is based on logs from university servers, which recorded various DoS attacks during the publicly available period.
In total, there are eighty columns within this dataset, each of which corresponds to an entry in the IDS logging system that the Unversity of New Brunswick has in place.

Each entry in the dataset is originally labeled with multiple classes, such as Benign, FTP-BruteForce, and Other. To simplify the task and formulate it as a binary classification problem, I will consolidate all non-Benign labels into a single Malicious category. This way, the model will learn to distinguish between Benign and Malicious network traffic.

## Import libraries and models
I import the libraries and modules that we will need for the project.

In [ ]:
import pandas as pd
import dask.dataframe as dd
from dask_ml.preprocessing import Categorizer
from sklearn.feature_selection import mutual_info_classif
from dask_ml.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import learning_curve
import matplotlib.pyplot as plt
import dask.array as da
import numpy as np
import os

RANDOM_STATE = 7

## Load and manipulate the dataset
I load the dataset using only a single large CSV instead of all the files.This simplifies processing, avoids mismatched type issues, and reduces memory usage.  
At the same time, I convert all labels that are not "Benign" into a single "Malicious" category.  
I use Dask that is a library which performs lazy loading to avoid loading the entire dataset into memory.

In [ ]:
def showPlot(width, height, title, xDescription, yDescription, values, color):
    plt.figure(figsize = (width, height))
    plt.bar(values.index, values.values, color = color)
    plt.title(title)
    plt.xlabel(xDescription)
    plt.ylabel(yDescription)
    plt.show()

def showCounts(dataset, feature, reindex, rename, width, height, title, xDescription, yDescription, color, showPlotFlag):
    counts = dataset[feature].value_counts(dropna = False).compute()
    if(rename != None):
        counts = counts.rename(index = rename)
    if(reindex != None):
        counts = counts.reindex(reindex)
    counts = counts.fillna(0)  
    counts = counts.astype(int)
    print(f"{counts}\nMalicious/Benign: {counts.get('Malicious', 0) / counts.get('Benign', 0):.2%}\n")
    if(showPlotFlag):
        showPlot(width, height, title, xDescription, yDescription, counts, color)

def showPurity(dataset, showPlotFlag):
    dirty_entries_dt = dataset[dataset.isna().any(axis = 1)]
    dirty_entries = dirty_entries_dt.shape[0].compute()
    total_entries = dataset.shape[0].compute()
    clean_entries = total_entries - dirty_entries
    print(f"Entries that have at least 1 null feature value: {dirty_entries}\nTotal Entries: {total_entries}\nPercentage of dirty entries: {dirty_entries / total_entries:.2%}\n")
    if(dirty_entries > 0):
        print("Analyzing dirty entries:\n" , dirty_entries_dt["Label"].value_counts().compute(), "\n")
    if(showPlotFlag):
        counts = pd.Series([clean_entries, dirty_entries], index = ["Clean Entries", "Dirty Entries"])
        showPlot(6, 4, "Clean Entries vs Dirty Entries", "Type", "Count", counts, ["skyblue", "brown"])    

dataset = dd.read_csv("Dataset/02-20-2018.csv", assume_missing = True)
dataset["Label"] = dataset["Label"].map(lambda x: "Malicious" if x != "Benign" else "Benign")
showCounts(dataset, "Label", ["Benign", "Malicious"], None, 6, 4, "Distribution of the Classes", "Class", "Count", ["green", "red"], True)

### Cleaning dirty entries
Now I check the entire dataset for entries with any missing feature values.

In [ ]:
showPurity(dataset, True)

At this stage, we observe that entries with missing values represent less than 1% of the overall dataset. Removing features or attempting to impute missing values is not appropriate at this point, since most features are important for traffic analysis and will be further examined during the subsequent analysis phase. Moreover, filling missing values in network traffic data would be unreliable and could introduce noise, potentially distorting the underlying patterns. For these reasons, I will remove the incomplete entries from the dataset.

Additionally, as previously noted, the dataset is heavily imbalanced toward Benign traffic, as it contains a significantly larger number of such samples. Therefore, removing these already limited “dirty” entries, which are all classified as Benign, is further justified and does not negatively impact the representativeness of the dataset.

In [ ]:
dataset = dataset.dropna()
showPurity(dataset, True)

### Analyzing infinite values
I analyze which features include infinite values across their entries.

In [ ]:
numeric_features = dataset.select_dtypes(include=[np.number])
features_inf = list(numeric_features.columns[np.isinf(numeric_features).any(axis = 0)])
print(f"Features containing infinite values: {features_inf}")

Now, I  check how many entries contain infinite values in those features.

In [ ]:
mask_inf = np.isinf(dataset[features_inf]).any(axis=1)
dataset_inf = dataset.loc[mask_inf]
showCounts(dataset_inf, "Label", ["Benign", "Malicious"], None, 6, 4, "Distribution of the Classes", "Class", "Count", ["green", "red"], True)

Since all detected samples containing infinite values belong to the benign class, and considering that the dataset is already strongly imbalanced, these entries can be safely removed without significantly affecting the representation of the minority (malicious) class.

In [ ]:
dataset = dataset.loc[~mask_inf]
showCounts(dataset, "Label", ["Benign", "Malicious"], None, 6, 4, "Distribution of the Classes", "Class", "Count", ["green", "red"], True)

## A Priori Feature Removal
This dataset contains network traffic analysis data between devices and university servers, so it is necessary to carefully examine all the features available in each entry and remove those that are not relevant for analyzing future network traffic and making predictions, thereby essentially avoiding model overfitting.  
### Showing features

In [ ]:
print(dataset.columns, "\n\nNumber of features: ", dataset.shape[1])

Among all these features, I can surely remove the <span style="color:red"><b>Flow ID</b></span>, which is unique and therefore carries no weight in the learning process. The <span style="color:red"><b>destination IP</b></span> is not relevant because it only indicates which university server the packets are sent to and does not help distinguish malicious from benign traffic. Similarly, the <span style="color:red"><b>source IP</b></span> is not useful, as it could be spoofed and, like the destination IP, does not aid in discrimination; in fact, it can cause overfitting—if the model learns that a specific IP is benign, it will continue to trust it blindly, which is very dangerous. The <span style="color:red"><b>source port</b></span> is also not important, since it is often random and, being frequently unknown, analyzing it usually makes little sense. On the other hand, the destination port could be useful because malicious attacks often target well-known ports, such as those for DNS, email services, or Other services, so analyzing it can contribute to traffic analysis and model learning. The <span style="color:red"><b>timestamp</b></span> could be useful in a time series study, but in our case, we remove it because it does not provide any relevant information.

In [ ]:
dataset = dataset.drop(["Flow ID", "Src IP", "Src Port", "Dst IP", "Timestamp"], axis = 1)
print(dataset.columns, "\n\nNumber of features: ", dataset.shape[1])

## Target Encoding
I analyze the types of the features.

In [ ]:
print(dataset.dtypes)

Before training the model, all input features are already in numerical format, and no categorical or textual variables are present in the dataset. The only categorical variable is the target label, which is encoded into a numerical representation prior to the train-test split to ensure compatibility with the learning algorithms.

In [ ]:
cat_enc = Categorizer(columns = ["Label"])
dataset = cat_enc.fit_transform(dataset)
dataset["Label"] = dataset["Label"].cat.codes
dataset["Label"] = dataset["Label"].astype(int)
print(dataset.dtypes,"\n")
showCounts(dataset, "Label", ["Benign", "Malicious"], {0: "Benign", 1: "Malicious"}, None, None, None, None, None, None, False)

## Create Training Set and Test set
Now we randomly split the dataset. As is well known, the training set is used to train the models, while the test set is used to evaluate the final performance on unseen data. In addition, a validation set is set aside and used during the training phase to monitor the model’s performance and support model selection and tuning. The random_state parameter, as the name suggests, is used to initialize the internal random number generator that determines how the data is split into training, validation, and test sets. According to the documentation, setting a random_state ensures that the split is reproducible. The validation set will therefore be used during training, while the test set is kept untouched until the final evaluation stage.

Since the dataset is managed using Dask, which processes data in chunks rather than loading it entirely into memory, I implemented a custom stratified split of the dataset. This choice was necessary because the library does not provide a built-in API comparable to that of scikit-learn for performing such operations.

In a simple yet effective way, the two classes were first separated into two different files. Then, for each file, each containing only one class, 70% of the data was assigned to the training set, 10% to the validation set and the remaining 20% to the test set. This approach ensures that each split maintains the same proportion of classes, improving the reliability and representativeness of the model evaluation.

I simulate a semi-supervised scenario in which 93% of the labels are unavailable in the training set; therefore, we create an additional copy of the training set where most of the labels have been removed (i.e., masked), while keeping only a small labeled subset. 

The supervised baseline is trained exclusively on the small labeled subset (7%), while the semi-supervised approach leverages both the labeled data and the remaining unlabeled portion (93%), enabling a fair comparison of their effectiveness under limited labeling conditions. It's applied the same previous approach so that preserves the class distribution across the splits

In [ ]:
def saveData(dataset, path):
    dataset.to_csv(path, index = False, single_file = True)

def divideDataset(dataset):
    benign = dataset[dataset["Label"] == 0]
    malicious = dataset[dataset["Label"] == 1]
    saveData(benign, "Dataset/benign_classes.csv")
    saveData(malicious, "Dataset/malicious_classes.csv")

def createTrainAndTestSetAndValidationSet(dataset_class1, dataset_class2):
    fractions = [0.7, 0.1, 0.2]
    fractions2 = [0.93, 0.07]
    train_set1, validation_set1, test_set1 = dataset_class1.random_split(fractions, random_state = RANDOM_STATE)
    train_set2, validation_set2, test_set2 = dataset_class2.random_split(fractions, random_state = RANDOM_STATE)
    train_set1_unlabeled, train_set1_labeled = train_set1.random_split(fractions2, random_state = RANDOM_STATE)
    train_set2_unlabeled, train_set2_labeled = train_set2.random_split(fractions2, random_state = RANDOM_STATE)
    train_set1_unlabeled["Label"] = -1
    train_set2_unlabeled["Label"] = -1
    train_set_only_labeled = dd.concat([train_set1_labeled, train_set2_labeled]) 
    train_set_only_labeled_shuffled = train_set_only_labeled.sample(frac = 1, random_state = RANDOM_STATE)  #trick to shuffle in Dask
    train_set_semi_labeled = dd.concat([train_set1_labeled, train_set2_labeled, train_set1_unlabeled, train_set2_unlabeled])
    train_set_semi_labeled_shuffled = train_set_semi_labeled.sample(frac = 1, random_state = RANDOM_STATE)  #trick to shuffle in Dask
    saveData(train_set_only_labeled_shuffled, "Dataset/train_set_only_labeled.csv")
    saveData(train_set_semi_labeled_shuffled, "Dataset/train_set_semi_labeled.csv")
    saveData(dd.concat([validation_set1, validation_set2]), "Dataset/validation_set.csv")
    saveData(dd.concat([test_set1, test_set2]), "Dataset/test_set.csv")
    
def stratified_train_test_split(dataset):
    if not (os.path.isfile("Dataset/train_set_only_labeled.csv") and os.path.isfile("Dataset/train_set_semi_labeled.csv") 
            and os.path.isfile("Dataset/test_set.csv") and os.path.isfile("Dataset/validation_set.csv")):
        divideDataset(dataset)
        benign = dd.read_csv("Dataset/benign_classes.csv")
        malicious = dd.read_csv("Dataset/malicious_classes.csv")
        createTrainAndTestSetAndValidationSet(benign, malicious)
    train_set_only_labeled = dd.read_csv("Dataset/train_set_only_labeled.csv", dtype = {"Label": "int32"})   
    train_set_semi_labeled = dd.read_csv("Dataset/train_set_semi_labeled.csv", dtype = {"Label": "int32"})   
    validation_set = dd.read_csv("Dataset/validation_set.csv", dtype = {"Label": "int32"}) 
    test_set = dd.read_csv("Dataset/test_set.csv", dtype = {"Label": "int32"}) 
    return train_set_only_labeled, train_set_semi_labeled, validation_set, test_set

train_set_only_labeled, train_set_semi_labeled, validation_set, test_set = stratified_train_test_split(dataset)
datasets = [("Only_labeled", train_set_only_labeled), ("Semi_labeled", train_set_semi_labeled), ("Validation", validation_set), ("Test", test_set)]
print("\n")
for name, dataset in datasets:
    print(f"{name}:")
    showCounts(dataset, "Label", ["Benign", "Malicious", "Unlabeled"], {0: "Benign", 1: "Malicious", -1: "Unlabeled"}, 6, 4, "Distribution of the Classes", "Class", "Count", ['green', 'red', 'grey'], True)

## Preparing data for the model
As I see, these datasets were split correctly. Now I prepare the data for machine learning algorithms.

### Balancing
As can be observed, both in the fully labeled training set and in the semi-labeled training set, there is a strong class imbalance, with a clear predominance of benign samples.

According to a statistical study conducted by Cloudflare, approximately 6.8% of Internet traffic is associated with malicious activities. In the dataset under analysis, malicious instances account for about 7.8% of the total samples, which is consistent with real-world conditions.

To address this imbalance, oversampling techniques could be applied by increasing the number of minority class examples (e.g., through duplication), but this may lead to overfitting. Alternatively, undersampling techniques can be used by reducing the number of majority class samples, at the cost of potentially losing some information.

In order to improve the model’s ability to learn and correctly classify both classes, undersampling of benign instances is applied. Specifically, the number of benign samples is reduced by one third rather than being brought down to the same level as the malicious ones. This choice helps mitigate the imbalance while avoiding an excessive loss of information.

The remaining imbalance is then handled during the modeling phase by assigning a higher weight to the minority (malicious) class, thereby improving the model’s ability to correctly identify it.

In [ ]:
def balancer(dataset, name):
    dataset_benign = dataset[dataset["Label"] == 0]
    dataset_malicious = dataset[dataset["Label"] != 0]
    fractionBalancer = 1/3
    dataset_benign = dataset_benign.sample(frac = fractionBalancer, random_state = RANDOM_STATE)
    dataset = dd.concat([dataset_benign, dataset_malicious])
    print(name)
    showCounts(dataset, "Label", ["Benign", "Malicious"], {0: "Benign", 1: "Malicious"}, 6, 4, "Distribution of the Classes", "Class", "Count", ['green', 'red'], True)
    return dataset

train_set_only_labeled = balancer(train_set_only_labeled, "Train_set_only_labeled")   
train_set_semi_labeled = balancer(train_set_semi_labeled, "Train_set_semi_labeled")

### Separation of input features and target labels
In this step, the datasets are split into input features (x) and target variables (y) for the training, validation, and test sets. This separation is performed for both the fully labeled dataset and the semi-supervised dataset in order to properly prepare the data for model training and evaluation.

In [ ]:
x_train_only_labeled = train_set_only_labeled.drop("Label", axis = 1)
y_train_only_labeled = train_set_only_labeled["Label"].copy()

x_train_semi_labeled = train_set_semi_labeled.drop("Label", axis = 1)
y_train_semi_labeled = train_set_semi_labeled["Label"].copy()

x_validation = validation_set.drop("Label", axis = 1)
y_validation = validation_set["Label"].copy()

x_test = test_set.drop("Label", axis = 1)
y_test = test_set["Label"].copy()

### Analyzing correlation
Correlation analysis was performed on the fully labeled subset, and the same feature relationships were assumed to hold for the semi-supervised dataset, which extends the original distribution with unlabeled samples.

In [ ]:
mutualInfoClassifier = mutual_info_classif(x_train_only_labeled, y_train_only_labeled)
mutualInfoClassifier_series = pd.Series(mutualInfoClassifier, index=x_train_only_labeled.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 20))
mutualInfoClassifier_series.sort_values().plot(kind="barh")
plt.title("Mutual Information by feature")
plt.xlabel("Percentual Score")
plt.ylabel("Features")
plt.tight_layout()
plt.show()

Now I will remove all features that have less than 5% correlation with the target, and then I will rescale all features using StandardScaler.

In [ ]:
x_train_only_labeled = x_train_only_labeled.drop(mutualInfoClassifier_series[mutualInfoClassifier_series < 0.05].index, axis = 1)
x_train_semi_labeled = x_train_semi_labeled.drop(mutualInfoClassifier_series[mutualInfoClassifier_series < 0.05].index, axis = 1)
x_validation = x_validation.drop(mutualInfoClassifier_series[mutualInfoClassifier_series < 0.05].index, axis = 1)
x_test = x_test.drop(mutualInfoClassifier_series[mutualInfoClassifier_series < 0.05].index, axis = 1)

print(x_train_only_labeled.columns, "\n\nNumber of features: ", x_train_only_labeled.shape[1])

Now I will scale all features using StandardScaler for both training sets.

In [ ]:
scaler1 = StandardScaler()
x_train_only_labeled = scaler1.fit_transform(x_train_only_labeled)
x_validation_only_labeled = scaler1.transform(x_validation)
x_test_only_labeled = scaler1.transform(x_test)

scaler2 = StandardScaler()
x_train_semi_labeled = scaler2.fit_transform(x_train_semi_labeled)
x_validation_semi_labeled = scaler2.transform(x_validation)
x_test_semi_labeled = scaler2.transform(x_test)

## Semi-Supervised learning approaches
Within the context of semi-supervised learning, several approaches have been proposed to effectively exploit unlabeled data alongside a limited set of labeled examples. One widely used technique is self-training, a wrapper-based method in which a model is initially trained on labeled data and then iteratively assigns pseudo-labels to unlabeled instances based on high-confidence predictions. These newly labeled samples are subsequently incorporated into the training set, allowing the model to progressively refine its performance.

In addition, graph-based methods such as Label Propagation and Label Spreading provide an alternative perspective. In these approaches, data points are represented as nodes within a graph, where edges encode similarities between instances. Label Propagation directly spreads label information across the graph structure, whereas Label Spreading introduces a regularization mechanism that improves robustness, particularly in the presence of noisy data.

Overall, these methods aim to leverage the underlying structure of the data distribution in order to enhance classification performance, especially in scenarios where labeled data is scarce. 

## Choice of the model and the approaches
As the classification model, I use Gradient Boosting, specifically the XGBoost algorithm, an ensemble method based on decision trees that are built sequentially to minimize a differentiable loss function. This approach is selected for its strong performance on structured data and its ability to model non-linear relationships between features. XGBoost is also suitable for large-scale datasets and integrates well with distributed computing frameworks such as Dask, enabling efficient training when data cannot be fully loaded into memory. Unlike Random Forests, it does not rely on out-of-bag error estimation and therefore requires an explicit validation strategy, such as a validation set or cross-validation, to monitor performance during training.

For completeness, the project also presents a comparison between a model trained using only 7% of the labeled data and models that additionally exploit the remaining 93% of unlabeled data. In particular, the latter are trained using three semi-supervised approaches: self-training, label propagation, and label spreading.  

From this point onward, the model training will be carried out using scikit-learn, as the processed datasets are of a reasonable size to be handled in memory (RAM). The work performed so far has been useful to simulate a more realistic scenario, where datasets are significantly larger and cannot be fully preprocessed or engineered in memory.

### Define the model
I do supervided learning

In [ ]:
def showConfusionMatrix(y_val, y_pred):
    cm = confusion_matrix(y_val, y_pred)   
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    plt.show()

model = LogisticRegression(class_weight = "balanced", max_iter = 1000)
model.fit(x_train_only_labeled, y_train_only_labeled)
y_val_pred = model.predict(x_validation_only_labeled)
report = classification_report(y_validation, y_val_pred)
print(report)
showConfusionMatrix(y_validation, y_val_pred)

### Controlled Feature Reduction to Assess the Impact of Semi-Supervised Learning
As can be observed, the logistic regression model already achieves very strong performance; this is mainly due to the fact that the dataset is highly clean and extensively pre-engineered, making it particularly well-suited for supervised learning models. 

However, in realistic semi-supervised learning (SSL) scenarios, datasets are typically characterized by limited labeled data and reduced informational richness, which makes the learning task significantly more challenging. 

For this reason, and in order to simulate a more realistic and less informative setting, the feature space is deliberately reduced. Specifically, only a subset of features is retained, selected based on both domain recommendations provided by the original source and correlation analysis, namely: 
<span style="color: green;"><b>Dst Port (Destination Port)</b></span>, 
<span style="color: green;"><b>Protocol</b></span>, 
<span style="color: green;"><b>Flow Duration</b></span>, 
<span style="color: green;"><b>Tot Fwd Pkts (Total Forward Packets)</b></span>, 
and 
<span style="color: green;"><b>Tot Bwd Pkts (Total Backward Packets)</b></span>. 

This reduction is not intended to artificially degrade the model, but rather to simulate conditions of limited and partially informative feature availability, under which the benefits of SSL approaches can be more meaningfully evaluated. In this context, the increased uncertainty of the supervised model allows us to better highlight how semi-supervised techniques can leverage unlabeled data to improve predictive performance.

In [ ]:
features = ["Dst Port", "Protocol", "Flow Duration", "Tot Fwd Pkts", "Tot Bwd Pkts"]
x_train_only_labeled = x_train_only_labeled[features]
x_validation_only_labeled = x_validation_only_labeled[features]
x_test_only_labeled = x_test_only_labeled[features]
x_train_semi_labeled = x_train_semi_labeled[features]
x_validation_semi_labeled = x_validation_semi_labeled[features]
x_test_semi_labeled = x_test_semi_labeled[features]

### Retrying to train the model
I retry supervised learning

In [ ]:
model.fit(x_train_only_labeled, y_train_only_labeled)
y_val_pred = model.predict(x_validation_only_labeled)
report = classification_report(y_validation, y_val_pred)
print(report)
showConfusionMatrix(y_validation, y_val_pred)